# TAMU API Quickstart for This Repo

This notebook shows how to connect to the TAMU-backed Azure OpenAI endpoint used by `LLM_PV`.
It is deliberately safe: it **masks the API key** instead of printing it, and it uses the same tested `chat_completions` path that works in this project.

What the key is:
- `TAMU_API_KEY` is your secret authentication token.
- It proves you are allowed to use the TAMU API.
- It is **not** the model name and it is **not** the endpoint URL.
- Never paste the raw key into a shared notebook, screenshot, or Git commit.

What this notebook tests:
1. Your environment variables can be loaded from `.env`.
2. The key and endpoint are present.
3. A direct Python API call succeeds.
4. The repo's `tamu_api_smoke.py` script succeeds too.


In [1]:
from __future__ import annotations

import json
import os
import subprocess
import sys
from pathlib import Path
from urllib.parse import urlparse

def find_project_root() -> Path:
    for candidate in [Path.cwd().resolve(), *Path.cwd().resolve().parents]:
        if (candidate / "program_synthesis" / "runner.py").exists():
            return candidate
    raise FileNotFoundError("Could not find the LLM_PV project root from the current working directory.")

def load_env_if_present(project_root: Path) -> Path | None:
    candidate_paths = [
        project_root / ".env",
        Path.cwd().resolve() / ".env",
        Path.cwd().resolve().parent / ".env",
    ]
    seen = set()
    for path in candidate_paths:
        path = path.resolve()
        if path in seen or not path.is_file():
            continue
        seen.add(path)
        for raw_line in path.read_text(encoding="utf-8").splitlines():
            line = raw_line.strip()
            if not line or line.startswith("#"):
                continue
            if line.startswith("export "):
                line = line[len("export "):].strip()
            if "=" not in line:
                continue
            key, value = line.split("=", 1)
            key = key.strip()
            value = value.strip()
            if len(value) >= 2 and value[0] == value[-1] and value[0] in {'"', "'"}:
                value = value[1:-1]
            os.environ.setdefault(key, value)
        return path
    return None

def env_first(*names: str, default: str = "") -> str:
    for name in names:
        value = os.getenv(name)
        if value and value.strip():
            return value.strip()
    return default

def mask_secret(secret: str) -> str:
    if not secret:
        return "<missing>"
    return f"<present: {len(secret)} chars>"

project_root = find_project_root()
loaded_env_path = load_env_if_present(project_root)
project_root


WindowsPath('C:/Users/pipin/Desktop/Coding/Personal/2026/CodeBoost/LLM_PV')

## Check the TAMU settings

This cell reads the values the repo uses and prints a **masked** summary.

Important distinction:
- `TAMU_API_KEY` is your secret.
- `TAMU_AZURE_ENDPOINT` is the server address.
- `OPENAI_MODEL=protected.gpt-5.2` is just the friendly alias used in this repo.
- The actual Azure deployment resolved by the repo is `gpt-5.2-deep-learning-fundamentals`.


In [2]:
deployment_aliases = {
    "protected.gpt-5.2": "gpt-5.2-deep-learning-fundamentals",
    "gpt-5.2": "gpt-5.2-deep-learning-fundamentals",
}

api_key = env_first("TAMU_API_KEY", "OPENAI_API_KEY", "TAMUS_AI_CHAT_API_KEY")
azure_endpoint = env_first("TAMU_AZURE_ENDPOINT", "AZURE_OPENAI_ENDPOINT", "TAMU_API_ENDPOINT", "DPF_URL")
api_version = env_first("TAMU_API_VERSION", "AZURE_OPENAI_API_VERSION", "OPENAI_API_VERSION", default="2024-12-01-preview")
requested_model = env_first("OPENAI_MODEL", default="protected.gpt-5.2")
api_mode = env_first("API_MODE", default="chat_completions")
deployment = env_first("TAMU_DEPLOYMENT", "TAMU_DEPLOYMENT_NAME", "AZURE_OPENAI_DEPLOYMENT", "AZURE_OPENAI_DEPLOYMENT_NAME")
deployment = deployment or deployment_aliases.get(requested_model, requested_model)

print("Loaded .env from        :", loaded_env_path or "<none>")
print("Project root            :", project_root)
print("TAMU_API_KEY            :", mask_secret(api_key))
print("TAMU_AZURE_ENDPOINT host:", urlparse(azure_endpoint).netloc or "<missing>")
print("TAMU_API_VERSION        :", api_version)
print("OPENAI_MODEL            :", requested_model)
print("Resolved deployment     :", deployment)
print("API_MODE                :", api_mode)

assert api_key, "Missing TAMU_API_KEY or OPENAI_API_KEY"
assert azure_endpoint, "Missing TAMU_AZURE_ENDPOINT"
assert api_mode == "chat_completions", "Use chat_completions for this repo setup"


Loaded .env from        : C:\Users\pipin\Desktop\Coding\Personal\2026\CodeBoost\LLM_PV\.env
Project root            : C:\Users\pipin\Desktop\Coding\Personal\2026\CodeBoost\LLM_PV
TAMU_API_KEY            : <present: 84 chars>
TAMU_AZURE_ENDPOINT host: tamu-it-ae-ai-prod-prod-eastus2.openai.azure.com
TAMU_API_VERSION        : 2024-12-01-preview
OPENAI_MODEL            : protected.gpt-5.2
Resolved deployment     : gpt-5.2-deep-learning-fundamentals
API_MODE                : chat_completions


## Direct Python connection test

This is the simplest working connection path in Python for the current repo setup. It uses `AzureOpenAI` with the TAMU endpoint and makes one tiny request.


In [3]:
from openai import AzureOpenAI

client = AzureOpenAI(
    api_key=api_key,
    azure_endpoint=azure_endpoint.rstrip("/"),
    api_version=api_version,
)

response = client.chat.completions.create(
    model=deployment,
    messages=[
        {"role": "user", "content": 'Return a compact JSON object exactly like {"ok": true}.'}
    ],
    max_completion_tokens=80,
)

text = response.choices[0].message.content
usage = getattr(response, "usage", None)

print("Connection worked.")
print("Response model :", getattr(response, "model", deployment))
print("Text preview   :", text)
if usage is not None:
    print("Prompt tokens  :", getattr(usage, "prompt_tokens", None))
    print("Completion toks:", getattr(usage, "completion_tokens", None))
    print("Total tokens   :", getattr(usage, "total_tokens", None))


Connection worked.
Response model : gpt-5.2-2025-12-11
Text preview   : {"ok": true}
Prompt tokens  : 18
Completion toks: 9
Total tokens   : 27


## Repo smoke test

This cell runs the project's existing `program_synthesis/boosted/tamu_api_smoke.py` script, then extracts the JSON result. That confirms your notebook setup and the repo's helper script agree with each other.


In [4]:
smoke_cmd = [
    sys.executable,
    str(project_root / "program_synthesis" / "boosted" / "tamu_api_smoke.py"),
    "--timeout",
    "90",
]

result = subprocess.run(
    smoke_cmd,
    cwd=project_root,
    capture_output=True,
    text=True,
    check=False,
)

combined_output = "\n".join(part for part in [result.stdout, result.stderr] if part and part.strip())
json_start = combined_output.find("{")
assert json_start >= 0, f"No JSON payload found in smoke output. Raw output:\n{combined_output}"
payload, _ = json.JSONDecoder().raw_decode(combined_output[json_start:])

print("Command     :", " ".join(smoke_cmd))
print("Return code :", result.returncode)
print(json.dumps(payload, indent=2))

assert result.returncode == 0, "Smoke test failed"
assert payload.get("ok") is True, "Smoke test did not return ok=true"


Command     : C:\Python313\python.exe C:\Users\pipin\Desktop\Coding\Personal\2026\CodeBoost\LLM_PV\program_synthesis\boosted\tamu_api_smoke.py --timeout 90
Return code : 0
{
  "ok": true,
  "client_type": "azure_openai",
  "api_mode": "chat_completions",
  "requested_model": "protected.gpt-5.2",
  "request_model": "gpt-5.2-deep-learning-fundamentals",
  "model": "gpt-5.2-2025-12-11",
  "usage": {
    "prompt_tokens": 18,
    "completion_tokens": 9,
    "total_tokens": 27,
    "prompt_tokens_details": {
      "cached_tokens": 0,
      "audio_tokens": 0
    },
    "completion_tokens_details": {
      "audio_tokens": 0,
      "reasoning_tokens": 0
    },
    "cached_tokens": 0,
    "reasoning_tokens": 0
  },
  "text_preview": "{\"ok\": true}"
}


## What to run next

If the cells above work, your TAMU connection is ready for the repo. A tiny real run is:

```bash
python program_synthesis/runner.py --functions fn_a --lengths 20 --attempts 1 --num-trials 1 --train-size 8 --val-size 8 --test-size 32 --timeout 120
```

That exact command already succeeded during verification while building this notebook.
